[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S03/Reto_S03/laboratorio_automl/S3.5_lab_autogluon.ipynb)

# S3.5 · Laboratorio de AutoML sin nube de pago (AutoGluon)
**Sprint 3 · Reto "Primer modelo en producción" · IFCD107**

Se ejecuta en Google Colab o en tu PC y **no tiene coste**.
Objetivo: predecir si una reserva de *TurisData Canarias* se cancelará.

Uso en Colab: ejecuta las celdas **de arriba abajo, de una en una** (Mayús+Intro). La primera instala AutoGluon (tarda 2–3 min).
Guía completa para trabajar en pareja: `sprints/S03/actividades/automl/guia_automl_en_parejas.md`.

## Instalación (solo Colab)
Si Colab os avisa de que hay que reiniciar el entorno, aceptad y seguid desde la celda **0. Configuración**.

In [1]:
!pip install -q "autogluon.tabular[lightgbm,xgboost,catboost]" fastapi httpx

## 0. Configuración

In [2]:
# Descarga de archivos del laboratorio (si no están ya en la carpeta de trabajo)
import os, urllib.request
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/sprints/S03/Reto_S03/laboratorio_automl/"
def archivo(nombre):
    """Devuelve el nombre del archivo; si no está en esta carpeta, lo descarga del repositorio del curso."""
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(BASE_URL + nombre, nombre)
    return nombre

import time

import numpy as np
import pandas as pd

RUTA_DATOS = archivo("reservas_turisdata.csv")
LIMITE_TIEMPO = 120   # segundos de entrenamiento (poned 15 para un ensayo rápido)
CARPETA_MODELO = "modelo_cancelaciones"
OBJETIVO = "cancelada"
SEMILLA = 42
np.random.seed(SEMILLA)

## 1. Cargar los datos

In [3]:
# Si no tienes el CSV, genéralo con:  python generar_reservas_turisdata.py
datos = pd.read_csv(RUTA_DATOS, parse_dates=["fecha_reserva", "fecha_llegada"])
print(datos.shape)
print(datos.head())

(6000, 18)
   id_reserva fecha_reserva fecha_llegada  antelacion_dias  noches  adultos  \
0        5760    2024-01-01    2024-03-21               80       3        3   
1        3674    2024-01-01    2024-06-06              157       4        1   
2        1488    2024-01-01    2024-03-19               78       7        2   
3        2259    2024-01-01    2024-03-07               66       7        3   
4        2442    2024-01-01    2024-06-23              174       6        2   

   ninos tipo_habitacion           regimen              canal pais_origen  \
0      2        estandar     media_pension            empresa          ES   
1      2        estandar          desayuno            directo          ES   
2      0        estandar  solo_alojamiento  web_intermediaria          ES   
3      0        familiar  solo_alojamiento  web_intermediaria          ES   
4      0        estandar          desayuno            agencia          ES   

            tarifa      deposito  precio_noche  pet

## 2. Exploración y calidad del dato (en una plataforma de nube: "explorar el dataset")

In [4]:
print("\nTipos de dato:\n", datos.dtypes)
print("\nNulos por columna:\n", datos.isna().sum()[datos.isna().sum() > 0])
print("\nProporción de cancelaciones:", round(datos[OBJETIVO].mean(), 3))
print("\nResumen numérico:\n", datos.describe().T[["mean", "std", "min", "max"]])
# Preguntas para el informe:
#  - ¿Está desbalanceada la clase objetivo? ¿Qué métrica conviene?
#  - ¿Qué columnas podrían provocar fuga de datos (información posterior al desenlace)?


Tipos de dato:
 id_reserva                        int64
fecha_reserva            datetime64[ns]
fecha_llegada            datetime64[ns]
antelacion_dias                   int64
noches                            int64
adultos                           int64
ninos                             int64
tipo_habitacion                  object
regimen                          object
canal                            object
pais_origen                      object
tarifa                           object
deposito                         object
precio_noche                    float64
peticiones_especiales             int64
cliente_repetidor                 int64
cancelaciones_previas             int64
cancelada                         int64
dtype: object

Nulos por columna:
 pais_origen    138
dtype: int64

Proporción de cancelaciones: 0.459

Resumen numérico:
                                              mean          std  \
id_reserva                                 3000.5  1732.195139   
fecha_re

## 3. Partición TEMPORAL (evita fuga de datos: se entrena con el pasado y se prueba con el futuro)

In [5]:
datos = datos.sort_values("fecha_reserva").reset_index(drop=True)
corte = int(len(datos) * 0.8)
entrenamiento = datos.iloc[:corte].drop(columns=["id_reserva"])
prueba = datos.iloc[corte:].drop(columns=["id_reserva"])
print(f"\nEntrenamiento: {len(entrenamiento)} filas ({entrenamiento.fecha_reserva.min().date()} a "
      f"{entrenamiento.fecha_reserva.max().date()})")
print(f"Prueba:        {len(prueba)} filas ({prueba.fecha_reserva.min().date()} a "
      f"{prueba.fecha_reserva.max().date()})")


Entrenamiento: 4800 filas (2024-01-01 a 2025-08-06)
Prueba:        1200 filas (2025-08-06 a 2025-12-30)


## 4. Entrenar con AutoML (equivale al experimento de Autopilot/Canvas)

In [6]:
from autogluon.tabular import TabularPredictor

inicio = time.time()
predictor = TabularPredictor(
    label=OBJETIVO,
    problem_type="binary",
    eval_metric="roc_auc",
    path=CARPETA_MODELO,
).fit(
    train_data=entrenamiento,
    time_limit=LIMITE_TIEMPO,
    presets="medium_quality",
)
tiempo_entrenamiento = time.time() - inicio
print(f"\nTiempo de entrenamiento: {tiempo_entrenamiento:.0f} s")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct 22 03:07:54 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.26 GB / 12.67 GB (88.8%)
Disk Space Avail:   86.69 GB / 107.72 GB (80.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/content/modelo_cancelaciones"
Train Data Rows:    4800
Train Data Columns: 16
Label Column:       cancelada
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    11490.66 MB
	Train Data (Original)


Tiempo de entrenamiento: 65 s


## 5. Revisar los modelos candidatos (leaderboard)

In [7]:
clasificacion = predictor.leaderboard(prueba, silent=True)
print(clasificacion[["model", "score_test", "score_val", "fit_time"]].head(10))
# Preguntas: ¿el mejor modelo es un ensamblado? ¿Cuánto tarda cada uno? ¿Merece la pena el coste de cómputo?

                 model  score_test  score_val   fit_time
0  WeightedEnsemble_L2    0.725488   0.786054  31.007567
1       NeuralNetTorch    0.720887   0.776906  14.931308
2             CatBoost    0.719310   0.776617   8.391376
3           LightGBMXT    0.717048   0.773594   7.620827
4             LightGBM    0.715486   0.763208   0.750096
5              XGBoost    0.712605   0.757050   0.968475
6       ExtraTreesEntr    0.704883   0.752074   1.878338
7       ExtraTreesGini    0.704783   0.751527   1.684080
8     RandomForestEntr    0.695675   0.739710   2.908495
9        LightGBMLarge    0.694259   0.750603   1.677891


## 6. Evaluar en el periodo de prueba

In [8]:
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

y_real = prueba[OBJETIVO]
prob = predictor.predict_proba(prueba.drop(columns=[OBJETIVO]))[1]  # probabilidad de cancelar
for umbral in (0.5, 0.3):
    pred = (prob >= umbral).astype(int)
    print(f"\nUmbral {umbral}: precisión={precision_score(y_real, pred):.3f} "
          f"recall={recall_score(y_real, pred):.3f} F1={f1_score(y_real, pred):.3f}")
print("\nROC-AUC:", round(roc_auc_score(y_real, prob), 3))
print("\nMatriz de confusión (umbral 0.5):\n", confusion_matrix(y_real, (prob >= 0.5).astype(int)))
print("\n", classification_report(y_real, (prob >= 0.5).astype(int), digits=3))
# Pregunta de negocio: ¿qué cuesta más, una cancelación no prevista o una alerta falsa? ¿Qué umbral elegirías?


Umbral 0.5: precisión=0.664 recall=0.562 F1=0.608

Umbral 0.3: precisión=0.578 recall=0.865 F1=0.693

ROC-AUC: 0.725

Matriz de confusión (umbral 0.5):
 [[459 164]
 [253 324]]

               precision    recall  f1-score   support

           0      0.645     0.737     0.688       623
           1      0.664     0.562     0.608       577

    accuracy                          0.652      1200
   macro avg      0.654     0.649     0.648      1200
weighted avg      0.654     0.652     0.650      1200



## 6b. Elegir el umbral con costes (EJEMPLO: cambia los dos importes por los vuestros)

In [9]:
COSTE_NO_PREVISTA = 150    # euros que cuesta una cancelación que no avisamos (noche vacía)
COSTE_ALERTA_FALSA = 30
 euros que cuesta una alerta que no hacía falta (p. ej. un descuento regalado)
y_array = y_real.to_numpy()
filas = []
for u in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    pred = (prob >= u).astype(int)
    no_previstas = int(((pred == 0) & (y_array == 1)).sum())
    falsas = int(((pred == 1) & (y_array == 0)).sum())
    filas.append({"umbral": u, "alertas": int(pred.sum()), "no_previstas": no_previstas,
                  "alertas_falsas": falsas, "coste_euros": COSTE_NO_PREVISTA * no_previstas + COSTE_ALERTA_FALSA * falsas})
tabla_costes = pd.DataFrame(filas)
print("\n", tabla_costes.to_string(index=False))
print("\nUmbral con menos coste (de los probados):", tabla_costes.loc[tabla_costes["coste_euros"].idxmin(), "umbral"])
print("Umbral teórico = alerta falsa / (alerta falsa + no prevista) =",
      round(COSTE_ALERTA_FALSA / (COSTE_ALERTA_FALSA + COSTE_NO_PREVISTA), 2))
print("Comparación: no avisar nunca =", COSTE_NO_PREVISTA * int(y_array.sum()), "euros · avisar a todos =",
      COSTE_ALERTA_FALSA * int((y_array == 0).sum()), "euros")
# Pregunta: ¿gana el modelo a «avisar a todos»? ¿Y si la alerta falsa costara 5 euros en vez de 40?


  umbral  alertas  no_previstas  alertas_falsas  coste_euros
    0.1     1168             2             593        18090
    0.2     1041            25             489        18420
    0.3      864            78             365        22650
    0.4      672           154             249        30570
    0.5      488           253             164        42870
    0.6      338           341             102        54210
    0.7      219           407              49        62520

Umbral con menos coste (de los probados): 0.1
Umbral teórico = alerta falsa / (alerta falsa + no prevista) = 0.17
Comparación: no avisar nunca = 86550 euros · avisar a todos = 18690 euros


## 7. Explicabilidad: importancia de variables

In [10]:
importancia = predictor.feature_importance(prueba, num_shuffle_sets=3, subsample_size=1000, silent=True)
print(importancia[["importance", "stddev"]].head(10))
# Pregunta: ¿tienen sentido las variables más importantes para el negocio? ¿Alguna sospechosa de fuga?

                       importance    stddev
antelacion_dias          0.067528  0.007577
deposito                 0.058266  0.013522
canal                    0.044163  0.011317
tarifa                   0.019615  0.003073
cancelaciones_previas    0.005350  0.000293
peticiones_especiales    0.004334  0.005788
noches                   0.003298  0.001346
tipo_habitacion          0.002812  0.000750
regimen                  0.001034  0.001470
precio_noche             0.000930  0.001519


## 8. Guardar y recargar el modelo (equivale a "desplegar"; el modelo queda en una carpeta)

In [11]:
predictor.save()
predictor_recargado = TabularPredictor.load(CARPETA_MODELO)
comprobacion = predictor_recargado.predict_proba(prueba.drop(columns=[OBJETIVO]).head(5))[1]
print("\nModelo recargado. Probabilidades de ejemplo:", comprobacion.round(3).tolist())

TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/modelo_cancelaciones")



Modelo recargado. Probabilidades de ejemplo: [0.358, 0.532, 0.468, 0.386, 0.138]


## 9. Función de predicción (la usará la API del paso 10)

In [12]:
def predecir_cancelacion(reserva: dict) -> dict:
    """Recibe una reserva (dict con las columnas de entrada) y devuelve la probabilidad de cancelación."""
    fila = pd.DataFrame([reserva])
    fila["fecha_reserva"] = pd.to_datetime(fila["fecha_reserva"])
    fila["fecha_llegada"] = pd.to_datetime(fila["fecha_llegada"])
    p = float(predictor_recargado.predict_proba(fila)[1].iloc[0])
    return {"probabilidad_cancelacion": round(p, 4), "alerta": p >= 0.5}


ejemplo = prueba.drop(columns=[OBJETIVO]).iloc[0].to_dict()
print("\nEjemplo de predicción:", predecir_cancelacion(ejemplo))


Ejemplo de predicción: {'probabilidad_cancelacion': 0.3579, 'alerta': False}


## 10. Servir el modelo por API (equivale al endpoint)

In [13]:
# En Colab no se puede abrir http://127.0.0.1:8000/docs desde tu navegador: usamos TestClient,
# que llama a la API "por dentro", igual que lo haría un cliente. En local: uvicorn api_cancelaciones:app --reload
import importlib, sys
archivo("api_cancelaciones.py")
sys.path.insert(0, os.getcwd())
api = importlib.import_module("api_cancelaciones")
from fastapi.testclient import TestClient
cliente = TestClient(api.app)
print(cliente.get("/salud").json())
reserva_ejemplo = {"fecha_reserva": "2025-11-03", "fecha_llegada": "2026-01-15", "antelacion_dias": 73, "noches": 5,
                   "adultos": 2, "ninos": 0, "tipo_habitacion": "estandar", "regimen": "desayuno",
                   "canal": "web_intermediaria", "pais_origen": "UK", "tarifa": "flexible",
                   "deposito": "sin_deposito", "precio_noche": 85.0, "peticiones_especiales": 0,
                   "cliente_repetidor": 0, "cancelaciones_previas": 0}
respuesta = cliente.post("/predecir", json=reserva_ejemplo)
print(respuesta.status_code, respuesta.json())
# Prueba también una reserva con datos imposibles (por ejemplo "noches": 0) y mira el error 422.

{'estado': 'ok', 'modelo': 'modelo_cancelaciones'}
200 {'probabilidad_cancelacion': 0.7887, 'alerta': True, 'umbral': 0.5}


## 11. Informe de recursos (equivale al "informe de costes"; aquí el coste económico es 0 EUR)

In [14]:
tamano_mb = sum(
    os.path.getsize(os.path.join(raiz, f))
    for raiz, _, ficheros in os.walk(CARPETA_MODELO) for f in ficheros
) / 1e6
print("\n--- Informe de recursos ---")
print(f"Tiempo de entrenamiento: {tiempo_entrenamiento:.0f} s (límite fijado: {LIMITE_TIEMPO} s)")
print(f"Tamaño del modelo en disco: {tamano_mb:.1f} MB")
print("Coste económico: 0 EUR (Colab gratuito o equipo local)")
print("Recursos que hay que cerrar: la sesión de Colab (Entorno de ejecución > Desconectar).")


--- Informe de recursos ---
Tiempo de entrenamiento: 65 s (límite fijado: 120 s)
Tamaño del modelo en disco: 226.1 MB
Coste económico: 0 EUR (Colab gratuito o equipo local)
Recursos que hay que cerrar: la sesión de Colab (Entorno de ejecución > Desconectar).
